In [1]:
%pip install ucimlrepo


[notice] A new release of pip is available: 24.3.1 -> 26.0
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import Libraries
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Any, List, Tuple, Dict,Any
"""
    Load the Adult Census Income dataset from UCI repository.
    Reference:
        https://github.com/uci-ml-repo/ucimlrepo/blob/main/src/demo.ipynb
    """
# Set pandas display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# 1. Data Loading 
def load_adult_dataset() -> pd.DataFrame:
    
    adult = fetch_ucirepo(id=2)
    
    if adult.data is None:
        raise ValueError("Failed to fetch dataset from UCI repository.")
    
    # Combine features and targets
    df = pd.concat([adult.data.features, adult.data.targets], axis=1)
    
    return df

#1.1 Explorating dataset function
#Any:any type is acceptable

def explore_dataset(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Returns:
        dict: Summary statistics including column types, shape, etc.
    """
    # Identify column types
    cat_cols = [col for col in df.columns if df[col].dtype == 'object']
    num_cols = [col for col in df.columns if df[col].dtype != 'object']
    
    summary = {
        'shape': df.shape,
        'categorical_columns': cat_cols,
        'numerical_columns': num_cols,
        'total_samples': df.shape[0],
        'total_features': df.shape[1]
    }
    
#Display overview
    print("1. DATASET OVERVIEW")
    display(df.head(10))
    print(f"\nColumn Data Types:")
    print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
    print(f"Numerical columns ({len(num_cols)}): {num_cols}")
    print(f"\nDataset Summary:")
    print(f"Total samples: {summary['total_samples']:,}")
    print(f"Total features: {summary['total_features']}")
    
    return summary

# Load and explore data
df = load_adult_dataset()
dataset_summary = explore_dataset(df)


1. DATASET OVERVIEW


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
5,37,Private,284582,Masters,14,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K
6,49,Private,160187,9th,5,Married-spouse-absent,Other-service,Not-in-family,Black,Female,0,0,16,Jamaica,<=50K
7,52,Self-emp-not-inc,209642,HS-grad,9,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,45,United-States,>50K
8,31,Private,45781,Masters,14,Never-married,Prof-specialty,Not-in-family,White,Female,14084,0,50,United-States,>50K
9,42,Private,159449,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,5178,0,40,United-States,>50K



Column Data Types:
Categorical columns (9): ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'native-country', 'income']
Numerical columns (6): ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Dataset Summary:
Total samples: 48,842
Total features: 15


In [3]:
#2. Cleanining missing values on dataset
"""
Clean missing values in the dataset by replacing specified placeholders with NaN.
if found any missing values then drop those rows.
Args:
df: Input DataFrame
    missing_value: Placeholder for missing values (default is '?')
    Returns:
        pd.DataFrame: Cleaned DataFrame with missing values handled
        Reference:
        https://www.geeksforgeeks.org/data-analysis/data-cleansing-introduction/
    """
def analyze_missing_values(df: pd.DataFrame, missing_value: str = '?') -> pd.DataFrame:
    
    print("2. MISSING VALUE ANALYSIS")

    missing_info = []
    
#Check object columns for missing value
    object_cols = df.select_dtypes(include=['object']).columns
    
    for col in object_cols:
        missing_count = (df[col] == missing_value).sum()
        if missing_count > 0:
            missing_pct = (missing_count / len(df)) * 100
            missing_info.append({
                'column': col,
                'missing_count': missing_count,
                'missing_percentage': missing_pct
            })
            print(f"{col}: {missing_count:,} ({missing_pct:.2f}%)")
    
#Check for NaN values
    nan_counts = df.isnull().sum()
    if nan_counts.sum() > 0:
        print("\nNaN values found:")
        print(nan_counts[nan_counts > 0])
    
    if not missing_info:
        print("No missing values found.")
        
    return pd.DataFrame(missing_info)

#Analyze missing values
missing_summary = analyze_missing_values(df)

2. MISSING VALUE ANALYSIS
workclass: 1,836 (3.76%)
occupation: 1,843 (3.77%)
native-country: 583 (1.19%)

NaN values found:
workclass         963
occupation        966
native-country    274
dtype: int64


In [5]:
#2.1 Data cleaning functions
"""
Remove columns not needed for demographic market analysis.     
Note:
 Columns removed FOR NOW for paper planner market analysis are:
 - 'fnlwgt': Census sampling weight (not useful for analysis)
 - 'education-num': Duplicate of 'education' column (numerical encoding)
    """

def remove_columns(df: pd.DataFrame, columns_to_remove: List[str]) -> pd.DataFrame:
   
        return df.drop(columns=columns_to_remove, errors='ignore')


def fix_categorical_inconsistencies(df: pd.DataFrame, missing_marker: str = '?') -> pd.DataFrame:
    """
    Fixign categorical inconsistencies for the 4 critical demographic variables.
    VANESSA CHECK THIS FUNCTION
    FOCUS: Only clean variables needed for our market analysis hypotheses:
        - age: Digital adoption patterns
        - sex: Gender-specific marketing
        - education: Professional planning needs
        - income: Purchasing power
    
    PROBLEM: whitespaces, punctuation, and missing values
                    
    Args: missing_marker: String representing missing values (default: '?')
        
    Returns: DataFrame with standardized critical demographics
    """
    # Focus ONLY on the 4 critical demographics for market analysis
    critical_demographics = ['age', 'sex', 'education', 'income']
    
    for col in critical_demographics:
        if col not in df.columns:
            continue
        
        # Only clean categorical (object) columns
        if df[col].dtype == 'object':
            # Strip whitespace, replace ?, remove periods
            df[col] = df[col].str.strip()
            df[col] = df[col].replace(missing_marker, np.nan)
            df[col] = df[col].str.rstrip('.')
    
    return df

def impute_missing_critical_demographics(df: pd.DataFrame, strategy: str = 'drop') -> pd.DataFrame:
    """
    Handle missing values in the 4 critical demographic variables.
    
    STRATEGY: Drop rows with missing critical demographics to ensure:
        - No bias from imputed demographic data
        - Accurate income levels (purchasing power)
        - Valid comparisons between demographic groups
    
    Args:
        df: Input DataFrame
        strategy: 'drop' (recommended) or 'mode'
        
    Returns:
        pd.DataFrame: DataFrame with complete critical demographic information
    """
    critical_demographics = ['age', 'sex', 'education', 'income']
    
    if strategy == 'drop':
        df = df.dropna(subset=critical_demographics)
    elif strategy == 'mode':
        for col in critical_demographics:
            if col in df.columns:
                missing_count = df[col].isnull().sum()
                if missing_count > 0:
                    mode_values = df[col].mode()
                    if len(mode_values) > 0:
                        df[col] = df[col].fillna(mode_values[0])
    
    return df

def normalize_income(df: pd.DataFrame) -> pd.DataFrame:
    """ This function converts income categories to binary:
        - <=50K → 0 (Lower purchasing power)
        - >50K  → 1 (Higher purchasing power) 
    Business Context: target higher-income 
    """
    if 'income' not in df.columns:
        raise ValueError("Income column not found in dataset")
    
    # Create binary income indicator
    df['income_binary'] = df['income'].apply(lambda x: 1 if '>50K' in str(x) else 0)
    
    return df

def remove_duplicates(df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    """
    Remove duplicate rows to ensure accurate market sizing.
    
    Returns:
        tuple: (cleaned DataFrame, number of duplicates removed)
    """
    duplicates_count = df.duplicated().sum()
    df_cleaned = df.drop_duplicates()
    
    return df_cleaned, duplicates_count




In [6]:
# 3. Execute Market-Focused Cleaning Pipeline

def clean_adult_dataset_for_market_analysis(df: pd.DataFrame, 
                                            columns_to_remove: List[str] = ['fnlwgt', 'education-num']) -> pd.DataFrame:
    
    print("3. DATA CLEANING PIPELINE\n")
   
    # Store initial state
    rows_before = len(df)
    df_cleaned = remove_columns(df, columns_to_remove)
    df_cleaned = fix_categorical_inconsistencies(df_cleaned)
    df_cleaned = impute_missing_critical_demographics(df_cleaned, strategy='drop')
    df_cleaned = normalize_income(df_cleaned)
    df_cleaned, duplicates_removed = remove_duplicates(df_cleaned)
    
    # Calculate 
    rows_after = len(df_cleaned)
    incomplete_removed = rows_before - rows_after - duplicates_removed
    data_retention = (rows_after / rows_before) * 100

    # CLEANED SUMMARY
    print(f" Columns removed: {len(columns_to_remove),} ({columns_to_remove})\n")
    print(f" Duplicates removed: {duplicates_removed:,}\n")
    print(f" Incomplete demographics removed: {incomplete_removed:,}\n")
    print(f" Final market sample: {rows_after:,} individuals\n")
    print(f" Data retention: {data_retention:.2f}% ({rows_after:,} of {rows_before:,} original records)\n")
    
    print("\n•••••••MARKET SEGMENTATION VARIABLES•••••••\n")
  
    if 'age' in df_cleaned.columns:
        print(f"Age:")
        print(f"  • Range: {df_cleaned['age'].min()}-{df_cleaned['age'].max()} years")
        print(f"  • Median: {df_cleaned['age'].median():.0f} years\n")
    
    if 'sex' in df_cleaned.columns:
        gender_dist = df_cleaned['sex'].value_counts()
        print(f"Gender:")
        for gender, count in gender_dist.items():
            pct = count / len(df_cleaned) * 100
            print(f"  • {gender}: {count:,} ({pct:.2f}%)")
        print()
    
    if 'education' in df_cleaned.columns:
        print(f"Education:")
        print(f"  • Total levels: {df_cleaned['education'].nunique()}")
        top_3 = df_cleaned['education'].value_counts().head(3)
        for edu, count in top_3.items():
            pct = count / len(df_cleaned) * 100
            print(f"  • {edu}: {count:,} ({pct:.2f}%)")
        print()
    
    if 'income_binary' in df_cleaned.columns:
        budget = (df_cleaned['income_binary'] == 0).sum()
        premium = (df_cleaned['income_binary'] == 1).sum()
        print(f"Income:")
        print(f"  • Budget market (≤$50K): {budget:,} ({budget/len(df_cleaned)*100:.2f}%)")
        print(f"  • Premium market (>$50K): {premium:,} ({premium/len(df_cleaned)*100:.2f}%)\n")
    
    return df_cleaned

# Execute cleaning pipeline
df_cleaned = clean_adult_dataset_for_market_analysis(df)

3. DATA CLEANING PIPELINE

 Columns removed: (2,) (['fnlwgt', 'education-num'])

 Duplicates removed: 6,281

 Incomplete demographics removed: 0

 Final market sample: 42,561 individuals

 Data retention: 87.14% (42,561 of 48,842 original records)


•••••••MARKET SEGMENTATION VARIABLES•••••••

Age:
  • Range: 17-90 years
  • Median: 38 years

Gender:
  • Male: 28,048 (65.90%)
  • Female: 14,513 (34.10%)

Education:
  • Total levels: 16
  • HS-grad: 12,944 (30.41%)
  • Some-college: 9,240 (21.71%)
  • Bachelors: 6,972 (16.38%)

Income:
  • Budget market (≤$50K): 32,111 (75.45%)
  • Premium market (>$50K): 10,450 (24.55%)



In [ ]:
"""
# Validate cleaned data

def validate_cleaned_data(df: pd.DataFrame) -> Dict[str, Any]:
    
    Validate the cleaned dataset.
    Returns:
        dict: Validation results
    
    print("5. CLEANING VALIDATION")
    
    validation = {
        'missing_values': df.isnull().sum().sum(),
        'duplicate_rows': df.duplicated().sum(),
        'shape': df.shape,
        'columns': list(df.columns)
    }
    
    print(f"\n✓ Final dataset shape: {validation['shape']}")
    print(f"✓ Missing values: {validation['missing_values']}")
    print(f"✓ Duplicate rows: {validation['duplicate_rows']}")
    
    # Check for whitespace issues
    print("✓ Checking categorical columns for whitespace issues:")
    
    has_whitespace = False
    
    #^\s]*? means is match any sequence of whitespace characters at the beginning | end of string
    #Reference:https://www.geeksforgeeks.org/python/regular-expression-python-examples/
    
    for col in df.select_dtypes(include='object').columns:
        # Use raw string for regex pattern
        whitespace_count = df[col].str.contains(r'^\s|\s$', regex=True, na=False).sum()
        if whitespace_count > 0:
            print(f"  Warning: {col} has {whitespace_count} values with whitespace")
            has_whitespace = True
    
    if not has_whitespace:
        print("  No whitespace issues found")
    
    # Display cleaned data sample
    print("\nCleaned data sample:")
    display(df.head(10))
    
    
    return validation

validation_results = validate_cleaned_data(df_cleaned)"""

Which demographic groups (age, gender, education, income) represent market opportunities for paper planners?


### Pandas Methods Used:

- df.drop() - Remove columns
- df.replace() - Replace values
- df.dropna() - Remove missing values
- df.str.strip() - Remove whitespace
- df.isnull() - Check for NaN
- df.value_counts() - Count unique values
- df.describe() - Summary statistics
- df.duplicated() - Check for duplicate rows
- df.drop_duplicates() - Remove duplicate rows

https://medium.com/@ugamakelechi501/how-to-prepare-and-clean-datasets-for-machine-learning-6ce2b7192d80



Columns that were removed: ['education-num', 'fnlwgt']

Reasons:
  - education-num: Duplicate of 'education' column (numerical encoding)
  - fnlwgt: Census sampling weigh


Future cleaning for a specific target such as age, gender,education...

Identification: For numerical columns like 'age', 'fnlwgt', 'capital-gain', 'capital-loss', and 'hours-per-week', we will need to identify extreme values that might be data entry errors or genuine but unusual observations 
. For example, an 'age' of 0 or an 'hours-per-week' of 1000 would be considered an outlier.


2. Standardizing Categorical Data
Categorical variables often contain inconsistencies that need standardization.

Inconsistent Entries: Check for variations in spelling, capitalization, or different representations of the same category (e.g., "United-States" vs. "United States" or "HS-grad" vs. "HS Grad"). The provided data consistently uses "United-States" and "HS-grad"